In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("E-commerce") \
    .master("local[*]") \
    .config("spark.driver.memory", "6g") \
    .config("spark.driver.maxResultSize", "2g") \
    .config("spark.sql.shuffle.partitions", "64") \
    .config("spark.default.parallelism", "4") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024)) \
    .config("spark.memory.fraction", "0.7") \
    .config("spark.memory.storageFraction", "0.3") \
    .config("spark.sql.files.maxPartitionBytes", "64MB") \
    .getOrCreate()


In [2]:
from pyspark.sql.functions import *

  .config("spark.driver.memory", "6g") \
    .config("spark.driver.maxResultSize", "3g") \
    .config("spark.sql.shuffle.partitions", "16") \
    .config("spark.default.parallelism", "8") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.autoBroadcastJoinThreshold", str(20 * 1024 * 1024)) \
    .config("spark.memory.fraction", "0.6") \
    .config("spark.memory.storageFraction", "0.3") \
    .config("spark.sql.files.maxPartitionBytes", "32MB") \

In [3]:
products = spark.read.csv(r"D:/VS CODE/.vscode/PDC_SEM_PROJECT/datasets/product_df.csv", header=True, inferSchema=True)
orders = spark.read.csv(r"D:/VS CODE/.vscode/PDC_SEM_PROJECT/datasets/order_df.csv", header=True, inferSchema=True)
order_items = spark.read.csv(r"D:/VS CODE/.vscode/PDC_SEM_PROJECT/datasets/order_item_df.csv", header=True, inferSchema=True)
customers = spark.read.csv(r"D:/VS CODE/.vscode/PDC_SEM_PROJECT/datasets/customer_df.csv", header=True, inferSchema=True)
payments = spark.read.csv(r"D:/VS CODE/.vscode/PDC_SEM_PROJECT/datasets/payment_df.csv", header=True, inferSchema=True)
categories = spark.read.csv(r"D:/VS CODE/.vscode/PDC_SEM_PROJECT/datasets/product_category_df.csv", header=True, inferSchema=True)
geolocation = spark.read.csv(r"D:/VS CODE/.vscode/PDC_SEM_PROJECT/datasets/geolocation_df.csv", header=True, inferSchema=True)
reviews = spark.read.csv(r"D:/VS CODE/.vscode/PDC_SEM_PROJECT/datasets/review_df.csv", header=True, inferSchema=True)
sellers = spark.read.csv(r"D:/VS CODE/.vscode/PDC_SEM_PROJECT/datasets/seller_df.csv", header=True, inferSchema=True)

# cache frequently used dataframes

In [4]:
orders.cache()
customers.cache()
order_items.cache()
products.cache()

DataFrame[product_id: string, product_name_lenght: int, product_description_lenght: int, product_photos_qty: int, product_weight_g: int, product_length_cm: int, product_height_cm: int, product_width_cm: int, product_category_name: string, product_size_category: string, product_name: string]

# apply joins 

In [5]:
orders_items_joined_df = orders.join(order_items,'order_id','inner')
orders_items_products_df = orders_items_joined_df.join(products,'product_id','inner')
orders_items_products_sellers_df = orders_items_products_df.join(broadcast(sellers),'seller_id','inner')
full_orders_df = orders_items_products_sellers_df.join(customers,'customer_id','inner')

full_orders_df = full_orders_df.join(broadcast(geolocation),full_orders_df.customer_zip_code_prefix == geolocation.geolocation_zip_code_prefix,'left')
full_orders_df = full_orders_df.join(broadcast(reviews),'order_id','left')
full_orders = full_orders_df.join(payments,'order_id','left')

In [6]:
full_orders.cache()

DataFrame[order_id: string, customer_id: string, seller_id: string, product_id: string, order_status: string, order_purchase_timestamp: timestamp, order_approved_at: date, order_delivered_carrier_date: date, order_delivered_customer_date: date, order_estimated_delivery_date: date, order_item_id: int, shipping_limit_date: timestamp, price: double, freight_value: double, product_name_lenght: int, product_description_lenght: int, product_photos_qty: int, product_weight_g: int, product_length_cm: int, product_height_cm: int, product_width_cm: int, product_category_name: string, product_size_category: string, product_name: string, seller_zip_code_prefix: int, seller_city: string, seller_state: string, customer_unique_id: string, customer_zip_code_prefix: int, customer_city: string, customer_state: string, geolocation_zip_code_prefix: int, geolocation_lat: double, geolocation_lng: double, geolocation_city: string, geolocation_state: string, review_id: string, review_score: string, review_com

# Total Orders Per Customer
# Average Review Score Per Seller
# Most Sold Products ( Top 10 )
# Top Customers By Spending

In [7]:
top_spending_customers = full_orders.groupBy('customer_id').agg(sum('price').alias('total_spent')).orderBy(desc('total_spent'))
top_spending_customers.show(5)

+--------------------+------------------+
|         customer_id|       total_spent|
+--------------------+------------------+
|63b964e79dee32a35...|         2501664.0|
|1ff773612ab8934db...|1658641.7999999512|
|de832e8dbb1f588a4...|1584990.5999999817|
|d72181923840c8895...|1488114.8999999566|
|904805ce22729b9b7...|         1319094.0|
+--------------------+------------------+
only showing top 5 rows



In [8]:
top_sold_products = full_orders.groupBy('product_id').agg(count('order_item_id').alias('total_sold')).orderBy(desc('total_sold'))

In [9]:
avg_rev_score_per_seller = full_orders.groupBy('seller_id').agg(avg('review_score').alias('avg_review_score'))
avg_rev_score_per_seller.show(5)

+--------------------+------------------+
|           seller_id|  avg_review_score|
+--------------------+------------------+
|16090f2ca825584b5...|4.1311950573024925|
|ff063b022a9a0aab9...| 3.985839440189542|
|8e6cc767478edae94...|3.8974387652387876|
|7139dc5186aa238b0...| 4.106708595387841|
|a49928bcdf77c55c6...|  2.98930255471884|
+--------------------+------------------+
only showing top 5 rows



In [10]:

total_orders_per_customer = full_orders.groupBy("customer_id").agg(count("order_id"))


# window functions

In [11]:
from pyspark.sql.window import Window

In [12]:
window_spec = Window.partitionBy("seller_id").orderBy(col("price").desc())


In [13]:
top_seller_products = full_orders.withColumn("rank", rank().over(window_spec)) \
                                 .filter(col("rank") <= 5)
top_seller_products = top_seller_products.select("seller_id", "price", "rank").show(5)

+--------------------+-----+----+
|           seller_id|price|rank|
+--------------------+-----+----+
|001cca7ae9ae17fb1...|199.0|   1|
|001cca7ae9ae17fb1...|199.0|   1|
|001cca7ae9ae17fb1...|199.0|   1|
|001cca7ae9ae17fb1...|199.0|   1|
|001cca7ae9ae17fb1...|199.0|   1|
+--------------------+-----+----+
only showing top 5 rows



# advance data enrichment and aggrigations 


In [14]:
# total rev and avg order value per customer
customer_spending = full_orders.groupBy("customer_id").agg(count("order_id").alias("total_orders"),
                                                      sum("price").alias("total_spent"),
                                                      round(avg("price"),2).alias("avg_order_value"))
customer_spending = customer_spending.orderBy(desc("total_spent"))
customer_spending.show(5)

+--------------------+------------+------------------+---------------+
|         customer_id|total_orders|       total_spent|avg_order_value|
+--------------------+------------+------------------+---------------+
|63b964e79dee32a35...|        6072|         2501664.0|          412.0|
|1ff773612ab8934db...|        5820|1658641.7999999512|         284.99|
|de832e8dbb1f588a4...|        2190|1584990.5999999817|         723.74|
|d72181923840c8895...|        2721|1488114.8999999566|          546.9|
|904805ce22729b9b7...|        3306|         1319094.0|          399.0|
+--------------------+------------+------------------+---------------+
only showing top 5 rows



In [15]:
# seller performance metric (revenue , averge review ,oredr count)
seller_perfromence = full_orders.groupBy("seller_id").agg(count("order_id").alias("total_orders"),
                                                      sum("price").alias("total_revenue"),
                                                      round(avg("review_score"),2).alias("avg_review_score"),
                                                      round(stddev("price"),2).alias("price_variability")).orderBy(desc("total_revenue"))
seller_perfromence.show(5)

+--------------------+------------+--------------------+----------------+-----------------+
|           seller_id|total_orders|       total_revenue|avg_review_score|price_variability|
+--------------------+------------+--------------------+----------------+-----------------+
|4869f7a5dfa277a7d...|      184319| 3.602987001001528E7|            4.09|           110.64|
|4a3ca9315b744ce9f...|      320935|3.2978693840039246E7|            3.78|            60.11|
|7c67e1448b00f6e96...|      231845|3.2107252050030634E7|            3.42|            50.36|
|da8622b14eb17ae28...|      260764|2.9523616630047645E7|            3.98|            73.26|
|fa1c13f2614d7b5c4...|       84723|2.6013136650007986E7|            4.38|           204.32|
+--------------------+------------+--------------------+----------------+-----------------+
only showing top 5 rows



In [16]:
# total popularity matrix
product_popularity = full_orders.groupBy("product_name").agg(count("order_item_id").alias("total_sold"),
                                                           sum("price").alias("total_revenue"),
                                                      round(avg("price"),2).alias("avg_price"),
                                                      collect_list('seller_id').alias('unique_sellers'),
                                                      round(avg("review_score"),2).alias("avg_review_score"),
                                                      round(stddev("price"),2).alias("price_variability")).orderBy(desc("total_sold"))
product_popularity.show(5)

+--------------------+----------+------------------+---------+--------------------+----------------+-----------------+
|        product_name|total_sold|     total_revenue|avg_price|      unique_sellers|avg_review_score|price_variability|
+--------------------+----------+------------------+---------+--------------------+----------------+-----------------+
|Compact Gadget #4571|     85348| 6067329.499998108|    71.09|[955fee9216a65b61...|            4.01|             3.19|
|  Premium Item #5033|     81048| 4583192.809999049|    56.55|[1f50f920176fa81d...|            3.83|            25.03|
|  Compact Item #2444|     77770|6831677.2099974165|    87.84|[4a3ca9315b744ce9...|             4.0|              4.1|
|Compact Gadget #6901|     60248| 3280533.130000288|    54.45|[1f50f920176fa81d...|            4.25|             4.37|
|  Sleek Gadget #7626|     59437| 8233515.030000735|   138.53|[a1043bafd471dff5...|            4.18|            16.79|
+--------------------+----------+---------------

In [17]:
from pyspark.sql.functions import date_format

monthly_trend = full_orders.groupBy(date_format("order_purchase_timestamp", "yyyy-MM").alias("month")) \
    .agg(
        sum("price").alias("monthly_revenue"),
        countDistinct("order_id").alias("order_count"),
        round(avg("price"), 2).alias("avg_order_value"),
        min("price").alias("min_order_value"),
        max("price").alias("max_order_value"),
    ) \
    .orderBy("monthly_revenue", ascending=False)

monthly_trend.show(12)

+-------+--------------------+-----------+---------------+---------------+---------------+
|  month|     monthly_revenue|order_count|avg_order_value|min_order_value|max_order_value|
+-------+--------------------+-----------+---------------+---------------+---------------+
|2017-11|1.4827464322999984E8|       7228|          108.5|           9.99|          865.9|
|2018-03|1.3762311341000003E8|       6935|         108.69|           9.99|         867.77|
|2018-01|1.3736151625000006E8|       6929|         107.21|           9.99|          889.9|
|2018-04|1.3497713536999995E8|       6726|         110.77|           9.99|          890.0|
|2018-05|1.3334703092000002E8|       6682|          111.8|           9.99|          889.2|
|2018-02|1.2500594166000004E8|       6507|         105.44|           9.99|          888.9|
|2018-06|1.2169386847000006E8|       6035|         111.82|           9.99|          890.0|
|2018-08|1.1693813599999996E8|       6319|         107.66|           9.99|          890.0|

In [18]:
customer_retention = full_orders.groupBy("customer_id").agg(first("order_purchase_timestamp").alias("first_order_date"),
                                                            last("order_purchase_timestamp").alias("last_order_date"),
                                                            count("order_id").alias("total_orders"),
                                                            round(avg("price"),2).alias("avg_order_value")).orderBy(desc("total_orders"))
customer_retention.show(5)

+--------------------+-------------------+-------------------+------------+---------------+
|         customer_id|   first_order_date|    last_order_date|total_orders|avg_order_value|
+--------------------+-------------------+-------------------+------------+---------------+
|50920f8cd0681fd86...|2018-01-27 11:28:32|2018-01-27 11:28:32|       10752|          43.82|
|5c87184371002d49e...|2018-01-05 19:15:37|2018-01-05 19:15:37|        6876|          12.49|
|d5f2b3f597c7ccafb...|2017-12-13 14:21:15|2017-12-13 14:21:15|        6706|           59.0|
|c2f18647725395af4...|2018-03-06 19:21:47|2018-03-06 19:21:47|        6612|           34.9|
|63b964e79dee32a35...|2018-02-14 16:34:27|2018-02-14 16:34:27|        6072|          412.0|
+--------------------+-------------------+-------------------+------------+---------------+
only showing top 5 rows



# extended enrichment

In [19]:
full_orders  = full_orders.withColumn('is_delivered', when(col('order_status') == 'delivered', 1).otherwise(0)) \
    .withColumn('is_canceled', when(col('order_status') == 'canceled', 1).otherwise(0))

In [20]:
full_orders.where(col('order_status')== 'canceled').select('order_id','customer_id','order_status').show(5)

+--------------------+--------------------+------------+
|            order_id|         customer_id|order_status|
+--------------------+--------------------+------------+
|00310b0c75bb13015...|0dad07848c618cc5a...|    canceled|
|00310b0c75bb13015...|0dad07848c618cc5a...|    canceled|
|00310b0c75bb13015...|0dad07848c618cc5a...|    canceled|
|00310b0c75bb13015...|0dad07848c618cc5a...|    canceled|
|00310b0c75bb13015...|0dad07848c618cc5a...|    canceled|
+--------------------+--------------------+------------+
only showing top 5 rows



In [21]:
# order rev calculation
full_orders = full_orders.withColumn('order_revenue', col('price') + col('freight_value'))
full_orders.select('price','freight_value','order_revenue').show(5)

+-----+-------------+-------------+
|price|freight_value|order_revenue|
+-----+-------------+-------------+
| 58.9|        13.29|        72.19|
| 58.9|        13.29|        72.19|
| 58.9|        13.29|        72.19|
| 58.9|        13.29|        72.19|
| 58.9|        13.29|        72.19|
+-----+-------------+-------------+
only showing top 5 rows



In [22]:
# customer segmentation based on spending
customer_spending = customer_spending.withColumn("customer_segment",
    when(col("avg_order_value") >= 1200, "high-value")
    .when((col("avg_order_value") < 1200) & (col("avg_order_value") >= 500), "medium-value")
    .otherwise("low-value")
)
customer_spending.show(5)

+--------------------+------------+------------------+---------------+----------------+
|         customer_id|total_orders|       total_spent|avg_order_value|customer_segment|
+--------------------+------------+------------------+---------------+----------------+
|63b964e79dee32a35...|        6072|         2501664.0|          412.0|       low-value|
|1ff773612ab8934db...|        5820|1658641.7999999512|         284.99|       low-value|
|de832e8dbb1f588a4...|        2190|1584990.5999999817|         723.74|    medium-value|
|d72181923840c8895...|        2721|1488114.8999999566|          546.9|    medium-value|
|904805ce22729b9b7...|        3306|         1319094.0|          399.0|       low-value|
+--------------------+------------+------------------+---------------+----------------+
only showing top 5 rows



In [23]:
full_orders = full_orders.join(customer_spending.select('customer_id','customer_segment'), 'customer_id', 'left')
full_orders.select('customer_id','customer_segment').show(5)

+--------------------+----------------+
|         customer_id|customer_segment|
+--------------------+----------------+
|3ce436f183e68e078...|       low-value|
|3ce436f183e68e078...|       low-value|
|3ce436f183e68e078...|       low-value|
|3ce436f183e68e078...|       low-value|
|3ce436f183e68e078...|       low-value|
+--------------------+----------------+
only showing top 5 rows



In [24]:
full_orders.select('order_purchase_timestamp').show(5)

+------------------------+
|order_purchase_timestamp|
+------------------------+
|     2017-09-13 08:59:02|
|     2017-09-13 08:59:02|
|     2017-09-13 08:59:02|
|     2017-09-13 08:59:02|
|     2017-09-13 08:59:02|
+------------------------+
only showing top 5 rows



In [25]:
# hourly distribution of orders

full_orders = full_orders.withColumn('hpur_of_day',expr("hour(order_purchase_timestamp)"))

In [26]:
# weekend vs weekday distribution of orders
full_orders = full_orders.withColumn('order_day_type',when(expr("dayofweek(order_purchase_timestamp) IN (1, 7)"), 'weekend').otherwise('weekday'))
full_orders.select('order_day_type').show(5)


+--------------+
|order_day_type|
+--------------+
|       weekday|
|       weekday|
|       weekday|
|       weekday|
|       weekday|
+--------------+
only showing top 5 rows



In [27]:
full_orders = full_orders.withColumn(
    "freight_category",
    when(col("freight_value") < 20, "low")
    .when((col("freight_value") >= 20) & (col("freight_value") < 50), "med")
    .otherwise("high")
)
full_orders.select("freight_value", "freight_category").show(5)

+-------------+----------------+
|freight_value|freight_category|
+-------------+----------------+
|        13.29|             low|
|        13.29|             low|
|        13.29|             low|
|        13.29|             low|
|        13.29|             low|
+-------------+----------------+
only showing top 5 rows



In [28]:

seller_rev = full_orders.groupBy('seller_id').agg(sum('price').alias('revenue'))
seller_rev.show(5)

+--------------------+------------------+
|           seller_id|           revenue|
+--------------------+------------------+
|16090f2ca825584b5...| 3190851.460000176|
|ff063b022a9a0aab9...|         1727524.0|
|8e6cc767478edae94...|1135610.9900000356|
|7139dc5186aa238b0...|          350436.0|
|a49928bcdf77c55c6...|1196351.6000000425|
+--------------------+------------------+
only showing top 5 rows



In [29]:

order_volume_by_state = full_orders.groupBy("customer_state").agg(countDistinct("order_id").alias("order_volume")) \
    .orderBy(desc("order_volume"))
order_volume_by_state.show(10)

+--------------+------------+
|customer_state|order_volume|
+--------------+------------+
|            SP|       40131|
|            RJ|       12336|
|            MG|       11182|
|            RS|        5263|
|            PR|        4821|
|            SC|        3486|
|            BA|        3249|
|            DF|        2055|
|            ES|        1979|
|            GO|        1934|
+--------------+------------+
only showing top 10 rows



saving files  and insights 

In [30]:
#full_orders = full_orders.toPandas()
#full_orders.to_csv(r"D:\VS CODE\.vscode\PDC_SEM_PROJECT\datasets\full_orders.csv", index=False)
customer_spending = customer_spending.toPandas()
customer_spending.to_csv(r"D:\VS CODE\.vscode\PDC_SEM_PROJECT\insights\customer_spending.csv", index=False)
customer_retention = customer_retention.toPandas()
customer_retention.to_csv(r"D:\VS CODE\.vscode\PDC_SEM_PROJECT\insights\customer_retention.csv", index=False)
product_popularity = product_popularity.toPandas()
product_popularity.to_csv(r"D:\VS CODE\.vscode\PDC_SEM_PROJECT\insights\product_popularity.csv", index=False)
monthly_trend = monthly_trend.toPandas()
monthly_trend.to_csv(r"D:\VS CODE\.vscode\PDC_SEM_PROJECT\insights\monthly_trend.csv", index=False)
top_sold_products = top_sold_products.toPandas()
top_sold_products.to_csv(r"D:\VS CODE\.vscode\PDC_SEM_PROJECT\insights\top_sold_products.csv", index=False)
top_spending_customers = top_spending_customers.toPandas()
top_spending_customers.to_csv(r"D:\VS CODE\.vscode\PDC_SEM_PROJECT\insights\top_spending_customers.csv", index=False)


In [31]:

seller_perfromence = seller_perfromence.toPandas()
seller_perfromence.to_csv(r"D:\VS CODE\.vscode\PDC_SEM_PROJECT\insights\seller_performance.csv", index=False)



In [32]:
full_orders = full_orders.dropDuplicates()

In [33]:
full_orders.write.mode("overwrite") \
    .option("header", True) \
    .option("compression", "gzip") \
    .csv(r"D:\VS CODE\.vscode\PDC_SEM_PROJECT\datasets\full_orders_compressed")


full_orders.coalesce(1).write.mode("overwrite") \
    .option("header", True) \
    .csv(r"D:\VS CODE\.vscode\PDC_SEM_PROJECT\datasets\full_orders.csv")


compressed_df = spark.read \
    .option("header", True) \
    .option("compression", "gzip") \
    .csv(r"D:\VS CODE\.vscode\PDC_SEM_PROJECT\datasets\full_orders_compressed")
